In [31]:
from tensorflow.keras.saving import save_model
import tensorflow as tf
from tensorflow.keras import Sequential, layers
import os
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np
import joblib

In [2]:
num_classes = 10

(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.astype('float32')
X_test  = X_test.astype('float32')
X_train /= 255
X_test  /= 255
y_train = to_categorical(y_train, num_classes)
y_test  = to_categorical(y_test, num_classes)

# These steps are new:
X_train = np.expand_dims(X_train, axis=3)
X_test  = np.expand_dims(X_test, axis=3)

In [6]:
train_set = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(len(X_train)).batch(batch_size=64)
test_set = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size=64)

In [7]:
model = Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(16, (3, 3), padding='same', activation='relu'),  # Add padding='same' here
    layers.MaxPooling2D(pool_size=(2, 2), padding='same'),           # Add padding='same' here
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

In [8]:
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.fit(
    train_set,
    epochs=5,
    verbose=1,
    validation_data=test_set
)

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 61s 62ms/step - accuracy: 0.3283 - loss: 1.7508 - val_accuracy: 0.3922 - val_loss: 1.5518
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 62s 66ms/step - accuracy: 0.4401 - loss: 1.4744 - val_accuracy: 0.5091 - val_loss: 1.3073
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 61s 65ms/step - accuracy: 0.5437 - loss: 1.2357 - val_accuracy: 0.5970 - val_loss: 1.1011
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 55s 58ms/step - accuracy: 0.5976 - loss: 1.0863 - val_accuracy: 0.6322 - val_loss: 0.9850
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 59s 63ms/step - accuracy: 0.6342 - loss: 0.9914 - val_accuracy: 0.6642 - val_loss: 0.9020


In [ ]:
joblib.dump(model, 'original_model.keras')

In [9]:
score = model.evaluate(X_test, y_test, verbose=0)
print('Test loss:',     score[0])
print('Test accuracy:', score[1])

Test loss: 0.9019892811775208
Test accuracy: 0.6642000079154968


In [16]:
def representative_data_gen():
  for input_value in tf.data.Dataset.from_tensor_slices(X_train).take(100):
    yield [input_value]  # Remove tf.cast and extra wrapping


In [25]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.representative_dataset = representative_data_gen
converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.target_spec.supported_types = [tf.float32]
converter.target_spec.supported_types = [tf.float16] # 50% size reduction, 2x latency reduction

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS
]

In [26]:
tflite_model = converter.convert()
model_name = 'model_float32.tflite'
with open(model_name, 'wb') as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\username\AppData\Local\Temp\tmpa4ig3u8x\assets


INFO:tensorflow:Assets written to: C:\Users\username\AppData\Local\Temp\tmpa4ig3u8x\assets


Saved artifact at 'C:\Users\username\AppData\Local\Temp\tmpa4ig3u8x'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor_7')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2420173962704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2420173963088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2420173962896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2420174636432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2420174636048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2420174636624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2420174635664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2420174637008: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [27]:
import numpy as np
import tensorflow as tf

# 1. Load the TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_path=model_name)
interpreter.allocate_tensors()

# 2. Get input and output tensor details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input shape: {input_details[0]['shape']}")
print(f"Input type: {input_details[0]['dtype']}")
print(f"Output shape: {output_details[0]['shape']}")

# 3. Prepare your input data - Keep as float32 (no casting to int8)
input_data = X_test[100:101].astype(np.float32)  # Shape: (1, 28, 28, 1)

# 4. Set the tensor and run inference
interpreter.set_tensor(input_details[0]['index'], input_data)
interpreter.invoke()

# 5. Extract the classification prediction
prediction = interpreter.get_tensor(output_details[0]['index'])
predicted_class = np.argmax(prediction[0])
actual_class = np.argmax(y_test[100])

print(f"\nPredicted Class: {predicted_class}")
print(f"Actual Class: {actual_class}")
print(f"Confidence: {prediction[0][predicted_class]:.4f}")
print(f"Match: {predicted_class == actual_class}")

# 6. Batch evaluation on test set
print("\n--- Evaluating on full test set ---")
correct = 0
for i in range(len(X_test)):
    input_data = X_test[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    prediction = interpreter.get_tensor(output_details[0]['index'])
    predicted_class = np.argmax(prediction[0])
    actual_class = np.argmax(y_test[i])
    if predicted_class == actual_class:
        correct += 1

accuracy = correct / len(X_test)
print(f"TFLite Model Accuracy: {accuracy:.4f}")

C:\Users\username\Projects\MachineLearning\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Input shape: [ 1 28 28  1]
Input type: <class 'numpy.float32'>
Output shape: [ 1 10]

Predicted Class: 6
Actual Class: 6
Confidence: 0.3677
Match: True

--- Evaluating on full test set ---
TFLite Model Accuracy: 0.6642


In [28]:
original_model_name = 'original_model.keras'
save_model(model, original_model_name)

In [29]:
os.path.getsize(model_name)

27404

In [30]:
os.path.getsize(original_model_name)

106509